<a href="https://colab.research.google.com/github/MalakMostafa-16003651/nyc-housing-transportation-analysis/blob/main/Person6_Research_Questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Person 6 (Lama): Research Questions 11 and 12

**RQ11: Which ZIP codes have high housing growth but poor road safety?** *(categorized scatter plot)*

**RQ12: Does the relationship between affordable housing production and serious crashes differ across boroughs?** *(scatter plot coloured by borough)*

**Input:** `df_integrated.csv`, Person 3's table with one row per `zip_code` + `year`. Both source datasets were aggregated to that level *before* merging:
- Housing: [Affordable Housing Production by Building](https://data.cityofnewyork.us/d/hg8x-zxpr) (`hg8x-zxpr`)
- Transportation: [Serious Injury Response, Tracking & Analysis Program (SIRTA)](https://data.cityofnewyork.us/d/xeqp-qz8h) (`xeqp-qz8h`)

**Website:** both charts are also live on the dashboard (`app/charts/rq11_rq12.py`). There they react to the borough, ZIP and year filters and to the housing and transportation dropdowns. This notebook uses the default measures: **affordable units** and **serious crashes**.

### Setup
Plotly, with the team's shared colours from `app/charts/style.py`, so the notebook and the website look the same.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from IPython.display import display

# Team colours: same meaning -> same colour in every chart (copied from app/charts/style.py)
COLOR_HOUSING = "#2a6fdb"   # blue   = affordable housing
COLOR_CRASH   = "#c43d5a"   # red    = serious crashes
COLOR_BOTH    = "#6a3d9a"   # purple = high on both
COLOR_OTHER   = "#c9ced3"   # grey   = context
BOROUGH_COLORS = {"Manhattan": "#2a6fdb", "Brooklyn": "#e0702b", "Queens": "#2f9e6e",
                  "Bronx": "#c43d5a", "Staten Island": "#7d5bbe"}


def style_fig(fig, title, x_title, y_title, height=560):
    """Common look: white template, left-aligned title, legend under the plot."""
    fig.update_layout(
        template="plotly_white", height=height,
        title=dict(text=title, x=0.01, xanchor="left", font=dict(size=17)),
        font=dict(family="Arial", size=13, color="#222"),
        legend=dict(orientation="h", yanchor="top", y=-0.18, x=0, title=None),
        margin=dict(l=70, r=30, t=80, b=40),
        hoverlabel=dict(bgcolor="white", font_size=12),
    )
    fig.update_xaxes(title_text=x_title)
    fig.update_yaxes(title_text=y_title)
    return fig

### Load the integrated data
The file is loaded from the repository root. In Colab, where the folder starts empty, the cell opens an upload button instead of crashing. ZIP codes are stored as 5-character text so `10456` never becomes `10456.0`.

In [2]:
from pathlib import Path

if not Path("df_integrated.csv").is_file():
    try:
        from google.colab import files          # only exists in Colab
        print("Please upload df_integrated.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("df_integrated.csv is missing — run from the repository root.")

df_integrated = pd.read_csv("df_integrated.csv", dtype={"zip_code": "string"})
df_integrated["zip_code"] = df_integrated["zip_code"].str.zfill(5)

# Sanity checks before analysis: the integration contract we rely on
assert not df_integrated.duplicated(["zip_code", "year"]).any(), "duplicate ZIP-year keys"
assert not df_integrated.isna().any().any(), "unexpected nulls"

print(df_integrated.shape, "| ZIPs:", df_integrated["zip_code"].nunique(),
      "| years:", df_integrated["year"].min(), "to", df_integrated["year"].max())
df_integrated.head()

(484, 11) | ZIPs: 143 | years: 2022 to 2026


,zip_code,year,housing_building_count,housing_project_count,affordable_units,total_housing_units,serious_crash_count,severe_injuries,non_severe_injuries,fatalities,borough
0,10002,2022,3.0,3.0,258.0,316.0,4.0,4.0,0.0,0.0,Manhattan
1,10011,2022,1.0,1.0,26.0,26.0,4.0,4.0,0.0,0.0,Manhattan
2,10018,2022,3.0,3.0,176.0,660.0,1.0,1.0,0.0,0.0,Manhattan
3,10023,2022,1.0,1.0,173.0,174.0,5.0,5.0,2.0,0.0,Manhattan
4,10024,2022,1.0,1.0,189.0,189.0,1.0,1.0,0.0,0.0,Manhattan


## Step 1: Why I compare **averages per matched year**, not totals

ZIP codes appear in `df_integrated` for different numbers of years. A ZIP is only included in a year when *both* datasets have records for it. If I add up all the years, a ZIP matched in five years automatically looks bigger on *both* housing and crashes than a ZIP matched in one year, just because it has more data.

The cell below measures this coverage effect. It counts how many years each ZIP is matched in. Then it checks how strongly the summed totals rank-correlate with that number (Spearman ρ).

In [3]:
zip_sums = (df_integrated.groupby("zip_code")
            .agg(matched_years=("year", "nunique"),
                 affordable_units=("affordable_units", "sum"),
                 serious_crashes=("serious_crash_count", "sum")))

print("ZIPs by number of matched years:", zip_sums["matched_years"].value_counts().sort_index().to_dict())
rho_units_years = stats.spearmanr(zip_sums["affordable_units"], zip_sums["matched_years"])[0]
rho_crash_years = stats.spearmanr(zip_sums["serious_crashes"], zip_sums["matched_years"])[0]
rho_sum_sum = stats.spearmanr(zip_sums["affordable_units"], zip_sums["serious_crashes"])[0]
print(f"Summed units   vs matched years: rho = {rho_units_years:.2f}")
print(f"Summed crashes vs matched years: rho = {rho_crash_years:.2f}")
print(f"Summed units   vs summed crashes: rho = {rho_sum_sum:.2f}")

ZIPs by number of matched years: {1: 21, 2: 28, 3: 21, 4: 21, 5: 52}
Summed units   vs matched years: rho = 0.76
Summed crashes vs matched years: rho = 0.80
Summed units   vs summed crashes: rho = 0.64


**What this shows:** both totals rise strongly with the number of matched years (ρ ≈ 0.76 and 0.80). Part of the strong totals-vs-totals correlation (ρ ≈ 0.64) is therefore just *data coverage*, not housing or crashes.

**My decision:** for RQ11 and RQ12, each ZIP's value is its **average per matched year**:

$$\text{affordable units per year} = \frac{\text{sum of affordable units}}{\text{number of matched years}}$$

This puts ZIPs with 1 year and 5 years of data on the same footing.
- *Alternative considered:* keeping only ZIPs matched in every year. That would throw away most ZIPs, because only 52 of 143 are matched in all five years, and 2022 and 2026 are partial years anyway. I use it below as a **robustness check** instead (Step 4).
- *Trade-off:* a ZIP matched in only one year has a noisier average. I report the number of matched years in the hover text so readers can see it.

In [4]:
def zip_year_averages(df, housing_col="affordable_units", safety_col="serious_crash_count"):
    """One row per ZIP: the two measures averaged over the ZIP's matched years.
    The borough is the ZIP's most common borough (a few ZIPs cross borough lines)."""
    return (df.groupby("zip_code", as_index=False)
              .agg(borough=("borough", lambda s: s.mode().iat[0]),
                   housing=(housing_col, "mean"),
                   safety=(safety_col, "mean"),
                   matched_years=("year", "nunique")))

per_zip = zip_year_averages(df_integrated)
per_zip.describe().round(1)

,housing,safety,matched_years
count,143.0,143.0,143.0
mean,192.6,11.1,3.4
std,193.1,6.6,1.5
min,2.0,1.0,1.0
25%,41.5,6.0,2.0
50%,134.5,10.5,4.0
75%,287.2,15.0,5.0
max,1000.2,39.5,5.0


**The data is heavily right-skewed.** Average affordable units per year range from about 2 to about 1,000, and the mean is much larger than the median. On normal axes most ZIPs would be squashed into one corner, so both charts use **log axes**. For the same reason I use the **Spearman rank correlation** (ρ). It depends only on the order of the values, so a few very large ZIPs cannot dominate it the way they would dominate Pearson's r.

## RQ11: Which ZIP codes have high housing growth but poor road safety?

**Definitions:**
- **High housing:** the ZIP's affordable units per year are at or above the **median ZIP**.
- **Poor road safety:** the ZIP's serious crashes per year are at or above the **median ZIP**.

Two median splits give **four groups**. The question's target group is *high housing + poor safety*.

*Why the median and not the top 25% (like RQ9)?* RQ9 already maps the extreme top-25% overlap. RQ11 asks a broader planning question: which growing areas have *worse than typical* road safety? The median is the natural line for "worse than typical". It also always splits the selection in half, so on the website the groups stay meaningful under any filter.

In [5]:
housing_cut = per_zip["housing"].median()
safety_cut = per_zip["safety"].median()
high_housing = per_zip["housing"] >= housing_cut
poor_safety = per_zip["safety"] >= safety_cut

TARGET = "High housing, poor safety"
per_zip["group"] = np.select(
    [high_housing & poor_safety, high_housing & ~poor_safety, ~high_housing & poor_safety],
    [TARGET, "High housing, safer", "Low housing, poor safety"],
    default="Low housing, safer")

print(f"Median ZIP: {housing_cut:.1f} affordable units/year, {safety_cut:.1f} serious crashes/year")
display(pd.crosstab(high_housing.map({True: "High housing", False: "Low housing"}),
                    poor_safety.map({True: "Poor safety", False: "Safer"})))

Median ZIP: 134.5 affordable units/year, 10.5 serious crashes/year


safety,Poor safety,Safer
housing,,
High housing,45,27
Low housing,27,44


### RQ11 chart: four groups of ZIP codes
Each dot is one ZIP code. The dashed lines are the medians. **Purple dots (top right)** answer the question. The three worst purple ZIPs are labelled. Hover over any dot to see its ZIP, borough and number of matched years.

In [6]:
GROUP_COLORS = {TARGET: COLOR_BOTH, "High housing, safer": COLOR_HOUSING,
                "Low housing, poor safety": COLOR_CRASH, "Low housing, safer": COLOR_OTHER}
counts = per_zip["group"].value_counts()
per_zip["group_label"] = per_zip["group"].map(lambda g: f"{g} ({counts[g]} ZIPs)")

fig_rq11 = px.scatter(
    per_zip, x="housing", y="safety", color="group_label", log_x=True, log_y=True,
    category_orders={"group_label": [f"{g} ({counts[g]} ZIPs)" for g in GROUP_COLORS]},
    color_discrete_map={f"{g} ({counts[g]} ZIPs)": c for g, c in GROUP_COLORS.items()},
    hover_data={"zip_code": True, "borough": True, "matched_years": True,
                "housing": ":.1f", "safety": ":.1f", "group_label": False},
    labels={"housing": "Affordable units per year", "safety": "Serious crashes per year",
            "zip_code": "ZIP code", "borough": "Borough", "matched_years": "Matched years"})
fig_rq11.update_traces(marker=dict(size=10, opacity=0.85, line=dict(width=0.5, color="white")))
fig_rq11.add_vline(x=housing_cut, line=dict(dash="dash", color="#888", width=1))
fig_rq11.add_hline(y=safety_cut, line=dict(dash="dash", color="#888", width=1))

# Label the 3 worst target ZIPs; arrows point different ways so labels never overlap.
# On log axes Plotly positions annotations in log10 units.
worst3 = per_zip[per_zip["group"] == TARGET].nlargest(3, "safety")
for (_, row), (ax, ay) in zip(worst3.iterrows(), [(35, -35), (-40, -30), (55, 15)]):
    fig_rq11.add_annotation(x=np.log10(row["housing"]), y=np.log10(row["safety"]),
                            text=row["zip_code"], ax=ax, ay=ay, arrowhead=0,
                            bgcolor="rgba(255,255,255,0.8)")

style_fig(fig_rq11, "RQ11: Which ZIP codes have high affordable housing but poor road safety?",
          "Affordable units per matched year (log scale)",
          "Serious crashes per matched year (log scale)")
fig_rq11.show()

### RQ11 table: the ZIPs that answer the question
The target group sorted by serious crashes per year, followed by how the group is spread across boroughs.

In [7]:
target = (per_zip[per_zip["group"] == TARGET]
          .sort_values("safety", ascending=False)
          [["zip_code", "borough", "housing", "safety", "matched_years"]]
          .rename(columns={"housing": "affordable_units_per_year",
                           "safety": "serious_crashes_per_year"})
          .reset_index(drop=True))
display(target.head(10).round(1))

by_borough = (per_zip.assign(is_target=per_zip["group"].eq(TARGET))
              .groupby("borough")
              .agg(zips=("zip_code", "count"), target_zips=("is_target", "sum")))
by_borough["share_of_borough_zips"] = (by_borough["target_zips"] / by_borough["zips"]).round(2)
display(by_borough.sort_values("target_zips", ascending=False))

# How "full" is the top-right corner compared with what independence would give?
expected_share = high_housing.mean() * poor_safety.mean()
print(f"Target group: {len(target)} of {len(per_zip)} ZIPs "
      f"({len(target)/len(per_zip):.0%}); if housing and safety were unrelated "
      f"we would expect about {expected_share:.0%}.")

,zip_code,borough,affordable_units_per_year,serious_crashes_per_year,matched_years
0,11207,Brooklyn,453.4,25.6,5
1,10019,Manhattan,180.0,25.5,2
2,11212,Brooklyn,539.6,23.8,5
3,11203,Brooklyn,269.8,23.6,5
4,10456,Bronx,593.6,21.8,5
5,10016,Manhattan,187.3,21.7,3
6,11206,Brooklyn,281.0,20.2,5
7,10467,Bronx,480.4,20.0,5
8,11226,Brooklyn,228.8,19.8,5
9,11101,Queens,354.2,19.6,5


,zips,target_zips,share_of_borough_zips
borough,,,
Brooklyn,37,18,0.49
Bronx,23,14,0.61
Manhattan,36,8,0.22
Queens,40,5,0.12
Staten Island,7,0,0.00


Target group: 45 of 143 ZIPs (31%); if housing and safety were unrelated we would expect about 25%.


### RQ11: Finding
- **45 of 143 matched ZIP codes (31%)** have above-median affordable housing production *and* above-median serious crashes per year. If the two measures were unrelated we would expect about 25%, so high housing and poor road safety co-occur somewhat more often than chance.
- The group is concentrated in **Brooklyn (18 ZIPs, about half of Brooklyn's matched ZIPs)** and **the Bronx (14 of 23 ZIPs)**. Manhattan has 8 and Queens 5. **Staten Island has none.**
- The ZIPs with the most serious crashes per year in this group are **11207 (East New York), 10019 (Midtown West), 11212 (Brownsville) and 11203 (East Flatbush)**. Each has about 24–26 serious crashes per year, roughly 2.3× the median ZIP. (10019 is matched in only 2 years, so its average is less certain.)
- **27 ZIPs have high housing but below-median crashes** (blue). So high affordable housing production does *not* automatically mean poor road safety. These ZIPs are a useful comparison group for planners. What do their streets do differently?

**Who could use this:** NYC DOT's Vision Zero programme and HPD could treat the purple ZIPs as a **shortlist for street-safety reviews near new affordable housing**. Thousands of new residents, including children and older adults, are moving into areas that already have above-typical serious crash counts.

**Honest limits:** these are *counts*, not risk rates (there is no population or traffic denominator). Being in the group does **not** mean housing caused the crashes. Busy, dense areas tend to have both more construction and more traffic. The median line is a choice, and ZIPs near it could switch groups with small changes.

## RQ12: Does the housing–crash relationship differ across boroughs?

For each borough separately, I measure how strongly a ZIP's affordable units per year go together with its serious crashes per year:
- **Spearman ρ** (strength of the monotonic relationship, from −1 to +1).
- **Log-log slope:** how much crashes rise when housing doubles (fitted on log10 values, so it reads as an elasticity).
- **A 95% bootstrap confidence interval for ρ**, because some boroughs have few ZIPs. Staten Island has only 7. I resample the borough's ZIPs 2,000 times with replacement and take the middle 95% of the ρ values.

In [8]:
rng = np.random.default_rng(42)          # fixed seed -> the same numbers every run

def bootstrap_rho_ci(x, y, n_boot=2000):
    """95% percentile bootstrap interval for Spearman rho (resampling ZIPs)."""
    x, y = np.asarray(x), np.asarray(y)
    rhos = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(x), len(x))
        if len(np.unique(x[idx])) > 1 and len(np.unique(y[idx])) > 1:
            rhos.append(stats.spearmanr(x[idx], y[idx])[0])
    return np.percentile(rhos, [2.5, 97.5])

rows = []
for borough, g in per_zip.groupby("borough"):
    rho = stats.spearmanr(g["housing"], g["safety"])[0]
    low, high = bootstrap_rho_ci(g["housing"], g["safety"])
    slope = np.polyfit(np.log10(g["housing"]), np.log10(g["safety"]), 1)[0]
    rows.append({"borough": borough, "zips": len(g), "spearman_rho": rho,
                 "rho_95ci_low": low, "rho_95ci_high": high, "log_log_slope": slope,
                 "median_units_per_year": g["housing"].median(),
                 "median_crashes_per_year": g["safety"].median()})
rq12_table = pd.DataFrame(rows).sort_values("spearman_rho", ascending=False).reset_index(drop=True)

overall_rho = stats.spearmanr(per_zip["housing"], per_zip["safety"])[0]
print(f"All boroughs together: rho = {overall_rho:.2f} across {len(per_zip)} ZIPs")
rq12_table.round(2)

All boroughs together: rho = 0.30 across 143 ZIPs


,borough,zips,spearman_rho,rho_95ci_low,rho_95ci_high,log_log_slope,median_units_per_year,median_crashes_per_year
0,Bronx,23,0.56,0.11,0.86,0.40,246.20,13.80
1,Queens,40,0.20,-0.12,0.50,0.09,36.70,7.00
2,Staten Island,7,0.19,-0.84,0.89,0.03,75.00,6.00
3,Manhattan,36,0.12,-0.22,0.45,0.07,146.38,8.68
4,Brooklyn,37,0.12,-0.28,0.46,-0.02,190.50,13.00


### RQ12 chart: one colour and one trend line per borough
Each dot is a ZIP. Each line is that borough's log-log trend. The legend shows each borough's ρ and number of ZIPs. **Click a borough in the legend to hide it, or double-click to isolate it.**

In [9]:
fig_rq12 = go.Figure()
for borough, color in BOROUGH_COLORS.items():
    g = per_zip[per_zip["borough"] == borough]
    rho = rq12_table.set_index("borough").loc[borough, "spearman_rho"]
    fig_rq12.add_trace(go.Scatter(
        x=g["housing"], y=g["safety"], mode="markers", legendgroup=borough,
        name=f"{borough} (ρ = {rho:.2f}, n = {len(g)})",
        marker=dict(color=color, size=9, opacity=0.8),
        customdata=np.stack([g["zip_code"], g["matched_years"]], axis=-1),
        hovertemplate=("<b>%{customdata[0]}</b><br>Affordable units/year: %{x:.1f}"
                       "<br>Serious crashes/year: %{y:.1f}<br>Matched years: %{customdata[1]}"
                       "<extra></extra>")))
    # Straight line on log axes = least squares fit on log10 values
    slope, intercept = np.polyfit(np.log10(g["housing"]), np.log10(g["safety"]), 1)
    xs = np.linspace(g["housing"].min(), g["housing"].max(), 50)
    fig_rq12.add_trace(go.Scatter(
        x=xs, y=10 ** (intercept + slope * np.log10(xs)), mode="lines", legendgroup=borough,
        showlegend=False, hoverinfo="skip", line=dict(color=color, width=2.5)))

fig_rq12.update_xaxes(type="log")
fig_rq12.update_yaxes(type="log")
style_fig(fig_rq12, "RQ12: Does the housing–crash relationship differ across boroughs?",
          "Affordable units per matched year (log scale)",
          "Serious crashes per matched year (log scale)")
fig_rq12.show()

### RQ12 summary chart: strength of the relationship with uncertainty
The scatter shows the shape. This chart compares the **strength** directly, with the bootstrap 95% interval as error bars. An interval that crosses 0 means the data cannot rule out *no* relationship in that borough.

In [10]:
ordered = rq12_table.sort_values("spearman_rho")
labels = [f"{b} (n = {n})" for b, n in zip(ordered["borough"], ordered["zips"])]
fig_rho = go.Figure(go.Bar(
    x=ordered["spearman_rho"], y=labels, orientation="h",
    marker_color=[BOROUGH_COLORS[b] for b in ordered["borough"]],
    error_x=dict(type="data", symmetric=False,
                 array=ordered["rho_95ci_high"] - ordered["spearman_rho"],
                 arrayminus=ordered["spearman_rho"] - ordered["rho_95ci_low"], color="#555"),
    hovertemplate="%{y}: ρ = %{x:.2f}<extra></extra>"))
# Values in a column on the right, so they never collide with bars or error bars
for label, rho in zip(labels, ordered["spearman_rho"]):
    fig_rho.add_annotation(x=1.02, y=label, text=f"ρ = {rho:.2f}", showarrow=False,
                           xanchor="left", font=dict(size=12))
fig_rho.add_vline(x=0, line=dict(color="#333", width=1))
fig_rho.add_vline(x=overall_rho, line=dict(color="#888", dash="dash", width=1),
                  annotation_text=f"All boroughs ρ = {overall_rho:.2f}", annotation_position="top")
style_fig(fig_rho, "RQ12: Strength of the housing–crash link by borough (95% bootstrap CI)",
          "Spearman ρ (affordable units/year vs serious crashes/year)", None, height=430)
fig_rho.update_xaxes(range=[-1, 1.2], tickvals=[-1, -0.5, 0, 0.5, 1])
fig_rho.show()

## Step 4: Robustness check with a balanced panel (2023–2025)
To check that the per-year averaging is not driving the result, I repeat RQ12 using only the ZIPs matched in **all three full calendar years 2023–2025**. That is the same balanced panel Person 5 used for RQ10. Every ZIP then has exactly three years, so plain totals are fair.

In [11]:
FULL_YEARS = [2023, 2024, 2025]
panel = df_integrated[df_integrated["year"].isin(FULL_YEARS)]
complete = panel.groupby("zip_code")["year"].nunique().loc[lambda s: s.eq(len(FULL_YEARS))].index
panel_zip = zip_year_averages(panel[panel["zip_code"].isin(complete)])

check = (panel_zip.groupby("borough")
         .apply(lambda g: pd.Series({"zips": len(g),
                                     "rho_balanced_panel": stats.spearmanr(g["housing"], g["safety"])[0]}),
                include_groups=False)
         .join(rq12_table.set_index("borough")["spearman_rho"].rename("rho_all_matched")))
print(f"Balanced panel: {len(panel_zip)} ZIPs, overall rho = "
      f"{stats.spearmanr(panel_zip['housing'], panel_zip['safety'])[0]:.2f}")
check.round(2)

Balanced panel: 72 ZIPs, overall rho = 0.32


,zips,rho_balanced_panel,rho_all_matched
borough,,,
Bronx,19.0,0.51,0.56
Brooklyn,28.0,0.21,0.12
Manhattan,13.0,0.26,0.12
Queens,12.0,0.04,0.20


### RQ12: Finding
- **Yes, the relationship differs by borough. It is only clear in the Bronx.** Across all 143 ZIPs, once ZIPs are compared per matched year, the link between affordable housing and serious crashes is **positive but modest (ρ ≈ 0.30)**.
- **Bronx: ρ ≈ 0.56**, and the 95% interval stays above zero. Bronx ZIPs that produce more affordable housing clearly tend to have more serious crashes.
- **Brooklyn, Manhattan and Queens: ρ ≈ 0.12–0.20**, and their intervals include zero. Within these boroughs, knowing a ZIP's housing production tells you little about its serious crashes. In Brooklyn, crash counts are high almost everywhere, not just in the high-housing ZIPs.
- **Staten Island (7 ZIPs)** is too small to draw a conclusion. Its interval is very wide.
- The **balanced-panel check** (72 ZIPs, 2023–2025) gives the same picture: the Bronx is strongest (ρ ≈ 0.51) and the other boroughs are weak. So the result is not caused by the averaging method.

**Why it matters:** the earlier citywide charts (RQ1, RQ4, RQ7) show strong correlations between *summed* housing and crash counts. RQ12 shows that a single citywide correlation hides big **borough differences**, and that part of the summed correlation comes from how many years each ZIP is matched. For policy, the Bronx is where new affordable housing and serious crashes most consistently go together. That makes it the clearest place to pair new housing with street-safety investment.

**Honest limits:** correlation is not causation. The Bronx pattern could reflect shared causes, such as wide arterial roads, density, or where land is available for building. Housing is dated by project start, not move-in. Crash records without ZIP codes were excluded upstream, and this may affect boroughs differently.

### Key numbers (rerun after any data change and check the text above still matches)

In [12]:
print("=== Coverage effect ===")
print(f"sum-vs-years rho: units {rho_units_years:.2f}, crashes {rho_crash_years:.2f}; "
      f"sum-vs-sum rho {rho_sum_sum:.2f}")
print("=== RQ11 ===")
print(f"medians: {housing_cut:.1f} units/yr, {safety_cut:.1f} crashes/yr")
print("groups:", per_zip["group"].value_counts().to_dict())
print("target by borough:", target["borough"].value_counts().to_dict())
print("top 5:", ", ".join(f"{z} ({b}, {c:.1f}/yr)" for z, b, c in
                          target[["zip_code", "borough", "serious_crashes_per_year"]].head(5).values))
print("=== RQ12 ===")
print(f"overall rho {overall_rho:.2f}")
for _, r in rq12_table.iterrows():
    print(f"{r.borough}: rho {r.spearman_rho:.2f} [{r.rho_95ci_low:.2f}, {r.rho_95ci_high:.2f}], n={r.zips}")

=== Coverage effect ===
sum-vs-years rho: units 0.76, crashes 0.80; sum-vs-sum rho 0.64
=== RQ11 ===
medians: 134.5 units/yr, 10.5 crashes/yr
groups: {'High housing, poor safety': 45, 'Low housing, safer': 44, 'Low housing, poor safety': 27, 'High housing, safer': 27}
target by borough: {'Brooklyn': 18, 'Bronx': 14, 'Manhattan': 8, 'Queens': 5}
top 5: 11207 (Brooklyn, 25.6/yr), 10019 (Manhattan, 25.5/yr), 11212 (Brooklyn, 23.8/yr), 11203 (Brooklyn, 23.6/yr), 10456 (Bronx, 21.8/yr)
=== RQ12 ===
overall rho 0.30
Bronx: rho 0.56 [0.11, 0.86], n=23
Queens: rho 0.20 [-0.12, 0.50], n=40
Staten Island: rho 0.19 [-0.84, 0.89], n=7
Manhattan: rho 0.12 [-0.22, 0.45], n=36
Brooklyn: rho 0.12 [-0.28, 0.46], n=37


### Check: notebook and website agree
If the repository's `app/` folder is present, this cell runs the **website's** RQ11 and RQ12 functions on the full data. It then confirms they plot the same per-ZIP values and borough correlations as this notebook. It is skipped in Colab.

In [13]:
import sys
if Path("app/charts/rq11_rq12.py").is_file():
    sys.path.insert(0, "app")
    from charts import rq11_rq12
    site_df = df_integrated.assign(housing_selected=df_integrated["affordable_units"],
                                   injuries_selected=df_integrated["serious_crash_count"],
                                   housing_label="Affordable units", transport_label="Serious crashes")
    site_x = sorted(x for t in rq11_rq12.rq11(site_df).data for x in t.x)
    assert np.allclose(site_x, sorted(per_zip["housing"])), "RQ11 values differ"
    legend = [t.name for t in rq11_rq12.rq12(site_df).data if t.showlegend is not False]
    for _, r in rq12_table.iterrows():
        assert any(n.startswith(f"{r.borough} (ρ = {r.spearman_rho:.2f}") for n in legend), r.borough
    print("PASS: website RQ11/RQ12 match the notebook.")
else:
    print("Skipped: app/ folder not available (e.g. Colab).")

PASS: website RQ11/RQ12 match the notebook.


### Prompt used (AI assistance)
The full prompt is recorded in `docs/AI_PROMPTS.md` under "Person 6 — Lama". In short: *"Create two graphs using our integrated dataset: (1) Which ZIP codes have high housing growth but poor road safety? A heatmap or categorized scatter plot. (2) Does the relationship between affordable housing production and serious crashes differ across boroughs? A scatter plot with different colours for each borough. Use Python, pandas and Plotly … don't assume something works until we've actually tested it."* Claude (Anthropic) helped write and test this notebook and the matching website charts. The analysis choices (per-year averaging, median split, Spearman with bootstrap intervals, balanced-panel check) are explained in the markdown above.